# Live tables

A random-walk ticker appends trades in the background; the session runs a cycle as soon as trades arrive, and each displayed table re-renders when a cycle changes it.

In [1]:
import random
import threading
import time

import kermodes as km
import pyarrow as pa

s = km.connect(":memory:")
ticks = s.input_table(
    {
        "sym": km.string(),
        "price": km.float64(),
        "size": km.int64(),
        "ts": km.timestamp(),
    }
)
stats = ticks.agg_by(
    "sym",
    n="count(*)",
    avg="mean(price)",
    vol="std(price)",
    low="min(price)",
    high="max(price)",
)
latest = ticks.where("size > 100").update("notional = price * size").last_by("sym")

In [2]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOG", "META", "TSLA", "AVGO", "JPM", "V"]
prices = dict.fromkeys(SYMBOLS, 100.0)
producing = threading.Event()


def produce() -> None:
    """Appends 200 trades every 50 ms until `producing` is cleared."""
    while producing.is_set():
        syms = random.choices(SYMBOLS, k=200)
        price = []
        for sym in syms:
            prices[sym] = max(0.01, prices[sym] + random.gauss(0, 0.05))
            price.append(round(prices[sym], 2))
        now = time.time_ns()
        ticks.append(
            pa.table(
                {
                    "sym": syms,
                    "price": price,
                    "size": [random.randint(1, 500) for _ in syms],
                    "ts": pa.array([now] * len(syms), pa.timestamp("ns", tz="UTC")),
                }
            )
        )
        time.sleep(0.05)


producing.set()
threading.Thread(target=produce, daemon=True).start()

In [3]:
stats

sym utf8,n int64,avg float64,vol float64,low float64,high float64
AMZN,1837780,133.17865017031366,21.050337255861507,74.69,179.32
TSLA,1837970,124.25495796449238,17.713314666414423,74.4,157.57
GOOG,1837944,63.28190438881622,14.069882299414921,30.64,104.17
AVGO,1840238,107.75341974788026,32.147700489389116,38.42,173.18
V,1838451,73.3717386430213,42.474686917567475,0.01,151.48
MSFT,1836976,158.87131456263012,30.398516860317006,96.83,221.18
JPM,1841450,84.6414815824502,10.266624604236354,53.66,110.46
NVDA,1839992,112.75296154005038,17.102600759861062,74.44,151.41
META,1837952,101.04128870068381,12.856479353509542,69.66,137.2
AAPL,1838847,155.50698309864816,30.05964031287747,85.0,202.06


In [4]:
latest

sym utf8,price float64,size int64,ts timestamp,notional float64
V,43.43,353,2026-11-11 15:59:07.162512,15330.789999999999
NVDA,79.52,335,2026-11-11 15:59:07.162512,26639.199999999997
TSLA,121.77,266,2026-11-11 15:59:07.162512,32390.82
AMZN,80.15,431,2026-11-11 15:59:07.162512,34544.65
AAPL,164.99,462,2026-11-11 15:59:07.162512,76225.38
META,124.38,182,2026-11-11 15:59:07.162512,22637.16
AVGO,76.22,318,2026-11-11 15:59:07.162512,24237.96
JPM,88.62,153,2026-11-11 15:59:07.162512,13558.86
MSFT,153.31,354,2026-11-11 15:59:07.162512,54271.74
GOOG,65.3,484,2026-11-11 15:59:07.162512,31605.199999999997


In [ ]:
producing.clear()  # stop the ticker; `s.stop()` stops the cycles